# Super Filter v3: Robust Street View invalid-image pipeline

This notebook replaces the brittle single-rule filtering approach with a layered multi-cue scoring system focused on low false positives for building-visible images.

What this version adds:
- central, tunable config for all thresholds and weights
- modular detectors (darkness, glare, blur, building/facade, open-scene, corridor-view, interior, occlusion, placeholder similarity, non-Google)
- hard-reject + soft-combined decision logic with rescue logic for building-visible borderline frames
- resumable processing for large runs (~20k images)
- CSV/JSON/JSONL outputs with per-image diagnostics and confidence
- invalid image routing into reason folders
- debug thumbnails and review sample manifests
- lightweight synthetic self-tests for sanity-checking core detectors


## STEP 1 - Existing pipeline review and failure diagnosis

### Current architecture in prior notebook
- Single-script Colab notebook with mostly hardcoded thresholds and per-image flags.
- Heavy reliance on DeepLab semantic segmentation for `building_ratio` with a single `building_ratio < 0.15` rule.
- Global blur via Laplacian variance only (`blur_score < 100`), no region awareness.
- Template match for Google logo in bottom-left without scale safety (caused OpenCV assertion crash when template > crop).
- Placeholder/no-imagery check using one average hash tolerance (`abs(hash - ref_hash) < 5`) only.
- CLIP zero-shot day/night and indoor/outdoor used as binary gate, not integrated with other cues.
- Any triggered detector copies image into category folder; no robust combined scoring policy.

### Why prior filters failed or overfired
1. **No-buildings overfired**: depended on segmentation ratio threshold and ignored structural facade cues; segmentation miss => false positives.
2. **Sky/trees/vehicle filters missed**: they were mostly removed or not integrated meaningfully in final logic.
3. **Blur missed many blurry images**: whole-image Laplacian variance is unstable when sky/flat regions dominate; no facade ROI.
4. **Street-corridor vs facade not handled**: no dedicated corridor score combining road center dominance + vanishing geometry + low side-facade presence.
5. **No-imagery exactness issue**: single hash tolerance can miss near-identical placeholders and has weak diagnostics.
6. **Interior/no-google noisy**: brittle logo/template assumptions and binary CLIP labels, with little outdoor evidence cross-check.
7. **Operational fragility**: no resumable manifest logic, minimal diagnostics, no confidence decomposition, and crash on logo template sizing.

### What is retained
- Bright/glare guardrail remains strong.
- Dark filter retained but tuned to prioritize clear-night cases.
- Placeholder-reference strategy retained but redesigned as multi-similarity (pHash/dHash/correlation).


## STEP 2 - Redesign plan before implementation

### New design
1. **Feature extraction layer (lightweight, OpenCV/NumPy)**
   - brightness/darkness, glare saturation/clipping, blur (ROI Laplacian + Tenengrad + edge density), building/facade structure, sky/vegetation/road masks, road-corridor geometry, occlusion, interior likelihood, placeholder similarity, non-google signal.
2. **Decision layer**
   - hard rejects for extreme unusable conditions (placeholder, extreme glare, extreme darkness, severe blur+low facade, strong interior).
   - weighted soft invalid score + synergy bonuses for combinations (low building + open scene, low building + corridor, blur + low facade, occlusion + low facade).
   - rescue logic to keep borderline frames when building evidence is strong (reduces false positives).
3. **Output/diagnostics layer**
   - per-image full score row with thresholds and contribution terms.
   - CSV + JSON + JSONL manifests.
   - optional invalid image copy/move into primary-reason subfolders.
   - debug overlays and review sampling manifests.
   - resumable processing using JSONL manifest.
4. **Validation layer**
   - synthetic detector sanity tests.
   - optional labeled evaluation helper if labels exist.


In [ ]:
# If needed in a fresh environment, uncomment:
# %pip install -q opencv-python pandas tqdm

from __future__ import annotations

import datetime as dt
import json
import logging
import math
import random
import shutil
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
# ----------------------------
# Configuration (centralized)
# ----------------------------

@dataclass
class ThresholdConfig:
    # darkness
    dark_mean_v: float = 38.0
    dark_low_ratio: float = 0.55
    dark_hard_mean_v: float = 24.0
    dark_hard_score: float = 0.90
    dark_soft_score: float = 0.72

    # glare / brightness
    glare_soft_score: float = 0.62
    glare_hard_score: float = 0.88
    bright_hard_pixel_ratio: float = 0.42
    clip_hard_ratio: float = 0.12

    # blur
    blur_soft_severity: float = 0.56
    blur_hard_severity: float = 0.78

    # building / facade
    building_min_score: float = 0.34
    building_soft_score: float = 0.42

    # open scene family
    sky_dominance: float = 0.45
    vegetation_dominance: float = 0.45
    road_corridor: float = 0.50
    open_scene_combined: float = 0.56

    # interior
    interior_soft: float = 0.72
    interior_hard: float = 0.84
    interior_outdoor_evidence_soft_max: float = 0.42
    interior_outdoor_evidence_hard_max: float = 0.35

    # occlusion / vehicle-like blocking
    occlusion_soft: float = 0.42
    occlusion_hard: float = 0.58

    # non-google (demoted signal)
    non_google_soft: float = 0.78

    # logo / template support
    logo_presence_expected: float = 0.55

    # placeholder similarity
    placeholder_soft_similarity: float = 0.88
    placeholder_hard_similarity: float = 0.94

    # final combined decision
    invalid_score_threshold: float = 0.62
    borderline_margin: float = 0.08


@dataclass
class WeightConfig:
    low_building: float = 0.34
    open_scene: float = 0.20
    blur: float = 0.14
    road_corridor: float = 0.10
    occlusion: float = 0.08
    interior: float = 0.10
    non_google: float = 0.03
    glare: float = 0.07
    dark: float = 0.04


@dataclass
class RuntimeConfig:
    max_side: int = 640
    recursive: bool = False
    supported_extensions: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

    invalid_action: str = "copy"  # copy | move | none
    organize_by_primary_reason: bool = True
    preserve_relative_structure: bool = True

    resume: bool = True
    random_seed: int = 1337
    log_level: str = "INFO"
    progress_every_n: int = 250

    save_debug_images: bool = True
    debug_max_per_reason: int = 20
    debug_save_valid_borderline: bool = True

    write_csv: bool = True
    write_json: bool = True
    write_jsonl: bool = True

    review_random_invalid_count: int = 40
    review_borderline_count: int = 30
    review_per_reason_count: int = 20
    review_copy_images: bool = False


@dataclass
class FilterConfig:
    thresholds: ThresholdConfig = field(default_factory=ThresholdConfig)
    weights: WeightConfig = field(default_factory=WeightConfig)
    runtime: RuntimeConfig = field(default_factory=RuntimeConfig)

    placeholder_reference_paths: List[str] = field(default_factory=list)
    logo_template_paths: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


DEFAULT_CONFIG = FilterConfig()


In [ ]:
# ----------------------------
# User inputs / run settings
# ----------------------------

# Optional Colab mount
try:
    from google.colab import drive  # type: ignore
except Exception:
    drive = None

# if drive is not None:
#     drive.mount('/content/drive')

INPUT_FOLDER = Path("/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Test_Images")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Filter_Output_v3")

CONFIG = FilterConfig(
    placeholder_reference_paths=[
        "/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/invalid_image.jpg"
    ],
    logo_template_paths=[
        # Add template paths if you have them, e.g.:
        # "/content/drive/MyDrive/.../google_logo_template.png"
    ],
)

# Optional targeted overrides for your first rerun
# CONFIG.thresholds.building_min_score = 0.33
# CONFIG.thresholds.invalid_score_threshold = 0.64
# CONFIG.runtime.invalid_action = "copy"

print("Input exists:", INPUT_FOLDER.exists())
print("Output exists:", OUTPUT_FOLDER.exists())
print("Placeholder refs:", CONFIG.placeholder_reference_paths)


In [ ]:
# ----------------------------
# Shared helpers
# ----------------------------

def clamp(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return float(max(low, min(high, value)))


def normalize(value: float, low: float, high: float) -> float:
    if high <= low:
        return 0.0
    return clamp((value - low) / (high - low))


def inverse_normalize(value: float, low: float, high: float) -> float:
    return 1.0 - normalize(value, low, high)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def discover_images(input_dir: Path, cfg: RuntimeConfig) -> List[Path]:
    suffixes = {s.lower() for s in cfg.supported_extensions}
    if cfg.recursive:
        candidates = [p for p in input_dir.rglob("*") if p.is_file()]
    else:
        candidates = [p for p in input_dir.iterdir() if p.is_file()]
    return sorted([p for p in candidates if p.suffix.lower() in suffixes])


def resize_keep_aspect(image: np.ndarray, max_side: int) -> Tuple[np.ndarray, float]:
    h, w = image.shape[:2]
    current_max = max(h, w)
    if current_max <= max_side:
        return image, 1.0
    scale = max_side / float(current_max)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized, scale


def compute_dhash(gray: np.ndarray, hash_size: int = 8) -> np.ndarray:
    tiny = cv2.resize(gray, (hash_size + 1, hash_size), interpolation=cv2.INTER_AREA)
    diff = tiny[:, 1:] > tiny[:, :-1]
    return diff.astype(np.uint8).reshape(-1)


def compute_phash(gray: np.ndarray, hash_size: int = 8, highfreq_factor: int = 4) -> np.ndarray:
    img_size = hash_size * highfreq_factor
    resized = cv2.resize(gray, (img_size, img_size), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(resized)
    low = dct[:hash_size, :hash_size]
    med = np.median(low[1:, 1:]) if hash_size > 1 else np.median(low)
    bits = low > med
    return bits.astype(np.uint8).reshape(-1)


def hash_similarity(bits_a: np.ndarray, bits_b: np.ndarray) -> float:
    if bits_a.shape != bits_b.shape or bits_a.size == 0:
        return 0.0
    dist = np.count_nonzero(bits_a != bits_b)
    return float(1.0 - dist / bits_a.size)


def corr_similarity(gray_a: np.ndarray, gray_b: np.ndarray, size: int = 128) -> float:
    a = cv2.resize(gray_a, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)
    b = cv2.resize(gray_b, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)

    a0 = a - a.mean()
    b0 = b - b.mean()
    denom = float(np.linalg.norm(a0) * np.linalg.norm(b0))

    if denom <= 1e-6:
        mad = float(np.mean(np.abs(a - b)))
        return inverse_normalize(mad, 0.0, 20.0)

    corr = float(np.sum(a0 * b0) / denom)
    return clamp((corr + 1.0) / 2.0)


def safe_template_match(crop: np.ndarray, template: np.ndarray) -> float:
    if crop is None or template is None:
        return 0.0
    ch, cw = crop.shape[:2]
    th, tw = template.shape[:2]
    if ch < 6 or cw < 6 or th < 6 or tw < 6:
        return 0.0

    templ = template
    if th > ch or tw > cw:
        scale = min(ch / float(th), cw / float(tw))
        if scale <= 0.2:
            return 0.0
        new_w = max(6, int(round(tw * scale)))
        new_h = max(6, int(round(th * scale)))
        templ = cv2.resize(template, (new_w, new_h), interpolation=cv2.INTER_AREA)

    th2, tw2 = templ.shape[:2]
    if th2 > ch or tw2 > cw:
        return 0.0

    res = cv2.matchTemplate(crop, templ, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, _ = cv2.minMaxLoc(res)
    return float(max_val)


def draw_debug_overlay(
    image_bgr: np.ndarray,
    decision: Dict[str, Any],
    scores: Dict[str, float],
) -> np.ndarray:
    canvas = image_bgr.copy()
    h, w = canvas.shape[:2]

    label = "INVALID" if decision["is_invalid"] else "VALID"
    color = (0, 0, 255) if decision["is_invalid"] else (40, 170, 40)

    lines = [
        f"{label} | primary={decision['primary_reason']} | conf={decision['confidence']:.2f}",
        f"invalid_score={decision['invalid_score']:.3f} hard={decision['hard_reject']} borderline={decision['borderline']}",
        f"building={scores['building_score']:.2f} open={scores['open_scene_score']:.2f} corridor={scores['road_corridor_score']:.2f}",
        f"blur={scores['blur_severity']:.2f} glare={scores['glare_score']:.2f} dark={scores['dark_score']:.2f}",
        f"interior={scores['interior_score']:.2f} occlusion={scores['occlusion_score']:.2f} placeholder={scores['placeholder_similarity']:.2f}",
        f"reasons={','.join(decision['reasons'])}",
    ]

    font = cv2.FONT_HERSHEY_SIMPLEX
    y = 22
    for line in lines:
        cv2.putText(canvas, line[:110], (10, y), font, 0.52, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(canvas, line[:110], (10, y), font, 0.52, (255, 255, 255), 1, cv2.LINE_AA)
        y += 20
        if y > h - 10:
            break

    cv2.rectangle(canvas, (0, 0), (w - 1, h - 1), color, 2)
    return canvas


In [ ]:
# ----------------------------
# Multi-cue detectors
# ----------------------------

class MultiCueDetectors:
    def __init__(self, config: FilterConfig):
        self.config = config
        self.placeholder_refs = self._load_placeholder_refs(config.placeholder_reference_paths)
        self.logo_templates = self._load_logo_templates(config.logo_template_paths)

    def _load_placeholder_refs(self, paths: List[str]) -> List[Dict[str, Any]]:
        refs: List[Dict[str, Any]] = []
        for p in paths:
            path = Path(p)
            if not path.exists():
                continue
            gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if gray is None:
                continue
            refs.append(
                {
                    "path": str(path),
                    "gray": gray,
                    "dhash": compute_dhash(gray),
                    "phash": compute_phash(gray),
                }
            )
        return refs

    def _load_logo_templates(self, paths: List[str]) -> List[np.ndarray]:
        templates: List[np.ndarray] = []
        for p in paths:
            path = Path(p)
            if not path.exists():
                continue
            gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                templates.append(gray)
        return templates

    def _compute_masks(self, hsv: np.ndarray, gray: np.ndarray) -> Dict[str, np.ndarray]:
        h, s, v = cv2.split(hsv)
        h = h.astype(np.int16)
        s = s.astype(np.int16)
        v = v.astype(np.int16)

        sky = (((h >= 85) & (h <= 130) & (s <= 140) & (v >= 70)) | ((s <= 25) & (v >= 210)))
        vegetation = (h >= 28) & (h <= 95) & (s >= 40) & (v >= 30)

        hh, ww = gray.shape
        lower = np.zeros((hh, ww), dtype=bool)
        lower[int(0.45 * hh) :, :] = True

        road = (s <= 60) & (v >= 45) & (v <= 200) & lower
        road = road & (~sky) & (~vegetation)

        kernel = np.ones((3, 3), np.uint8)
        sky = cv2.morphologyEx(sky.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=1).astype(bool)
        vegetation = cv2.morphologyEx(vegetation.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=1).astype(bool)
        road = cv2.morphologyEx(road.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=1).astype(bool)

        facade = (~sky) & (~vegetation) & (gray > 28)

        return {
            "sky": sky,
            "vegetation": vegetation,
            "road": road,
            "facade_candidate": facade,
        }

    def _line_features(self, canny: np.ndarray, mask: Optional[np.ndarray] = None) -> Dict[str, float]:
        h, w = canny.shape
        working = canny.copy()
        if mask is not None:
            working = cv2.bitwise_and(working, working, mask=mask.astype(np.uint8) * 255)

        min_len = max(24, int(0.08 * min(h, w)))
        lines = cv2.HoughLinesP(
            working,
            rho=1,
            theta=np.pi / 180,
            threshold=50,
            minLineLength=min_len,
            maxLineGap=8,
        )

        if lines is None:
            return {
                "line_count": 0.0,
                "horizontal_ratio": 0.0,
                "vertical_ratio": 0.0,
                "diagonal_ratio": 0.0,
                "hv_ratio": 0.0,
                "vanish_like": 0.0,
            }

        total = float(len(lines))
        horizontal = 0
        vertical = 0
        diagonal = 0
        vanish_like = 0

        for line in lines[:, 0, :]:
            x1, y1, x2, y2 = [int(v) for v in line]
            dx = x2 - x1
            dy = y2 - y1
            length = math.hypot(dx, dy)
            if length < min_len:
                continue

            angle = abs(math.degrees(math.atan2(dy, dx)))
            if angle < 20 or angle > 160:
                horizontal += 1
            elif 70 < angle < 110:
                vertical += 1
            elif 20 <= angle <= 70 or 110 <= angle <= 160:
                diagonal += 1

            top_y = min(y1, y2)
            bottom_y = max(y1, y2)
            mid_x = 0.5 * (x1 + x2)
            if (
                bottom_y >= 0.85 * h
                and top_y <= 0.65 * h
                and abs(mid_x - (w / 2.0)) <= 0.35 * w
                and (20 <= angle <= 70 or 110 <= angle <= 160)
            ):
                vanish_like += 1

        hv_ratio = (horizontal + vertical) / max(total, 1.0)

        return {
            "line_count": total,
            "horizontal_ratio": horizontal / max(total, 1.0),
            "vertical_ratio": vertical / max(total, 1.0),
            "diagonal_ratio": diagonal / max(total, 1.0),
            "hv_ratio": hv_ratio,
            "vanish_like": vanish_like / max(total, 1.0),
        }

    def _score_darkness(self, v_channel: np.ndarray) -> Dict[str, float]:
        mean_v = float(v_channel.mean())
        low_ratio = float(np.mean(v_channel < 40))
        very_low_ratio = float(np.mean(v_channel < 20))

        score = (
            0.55 * inverse_normalize(mean_v, 25.0, 95.0)
            + 0.35 * normalize(low_ratio, 0.20, 0.92)
            + 0.10 * normalize(very_low_ratio, 0.05, 0.70)
        )
        return {
            "brightness_mean_v": mean_v,
            "dark_low_ratio": low_ratio,
            "dark_very_low_ratio": very_low_ratio,
            "dark_score": clamp(score),
        }

    def _score_glare(self, bgr: np.ndarray, hsv: np.ndarray, gray: np.ndarray) -> Dict[str, float]:
        v = hsv[:, :, 2]
        s = hsv[:, :, 1]
        max_channel = np.max(bgr, axis=2)

        bright_mask = v > 245
        bright_ratio = float(np.mean(bright_mask))
        clip_ratio = float(np.mean(max_channel > 250))
        bright_low_sat_ratio = float(np.mean((v > 240) & (s < 30)))

        local_std = float(np.std(gray[bright_mask])) if np.any(bright_mask) else 50.0
        washout_score = inverse_normalize(local_std, 10.0, 45.0)

        hotspot_ratio = 0.0
        if np.any(bright_mask):
            num, _, stats, _ = cv2.connectedComponentsWithStats(bright_mask.astype(np.uint8), connectivity=8)
            if num > 1:
                hotspot_ratio = float(np.max(stats[1:, cv2.CC_STAT_AREA]) / bright_mask.size)

        glare_score = (
            0.38 * normalize(bright_ratio, 0.02, 0.55)
            + 0.24 * normalize(clip_ratio, 0.01, 0.20)
            + 0.20 * normalize(bright_low_sat_ratio, 0.01, 0.35)
            + 0.18 * max(washout_score, normalize(hotspot_ratio, 0.02, 0.40))
        )

        return {
            "bright_pixel_ratio": bright_ratio,
            "clip_ratio": clip_ratio,
            "bright_low_sat_ratio": bright_low_sat_ratio,
            "glare_hotspot_ratio": hotspot_ratio,
            "glare_score": clamp(glare_score),
        }

    def _score_blur(
        self,
        gray: np.ndarray,
        canny: np.ndarray,
        facade_mask: np.ndarray,
    ) -> Dict[str, float]:
        h, w = gray.shape
        roi = facade_mask.copy()
        roi[: int(0.10 * h), :] = False
        roi[int(0.93 * h) :, :] = False

        if float(np.mean(roi)) < 0.06:
            roi = np.zeros_like(facade_mask, dtype=bool)
            roi[int(0.20 * h) : int(0.90 * h), int(0.10 * w) : int(0.90 * w)] = True

        lap = cv2.Laplacian(gray, cv2.CV_32F)
        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        grad_mag = cv2.magnitude(gx, gy)

        roi_values = roi if np.any(roi) else np.ones_like(roi, dtype=bool)

        lap_var = float(np.var(lap[roi_values]))
        tenengrad_mean = float(np.mean(grad_mag[roi_values]))
        edge_density = float(np.mean(canny[roi_values] > 0))

        blur_severity = (
            0.46 * inverse_normalize(lap_var, 45.0, 250.0)
            + 0.34 * inverse_normalize(tenengrad_mean, 8.0, 30.0)
            + 0.20 * inverse_normalize(edge_density, 0.04, 0.20)
        )

        return {
            "laplacian_var_roi": lap_var,
            "tenengrad_mean_roi": tenengrad_mean,
            "edge_density_roi": edge_density,
            "blur_severity": clamp(blur_severity),
        }

    def _score_building_presence(
        self,
        gray: np.ndarray,
        canny: np.ndarray,
        facade_mask: np.ndarray,
    ) -> Dict[str, float]:
        h, w = gray.shape

        mid_band = np.zeros_like(facade_mask, dtype=bool)
        mid_band[int(0.12 * h) : int(0.88 * h), :] = True

        candidate = facade_mask & mid_band
        candidate_ratio = float(np.mean(candidate))

        if np.any(candidate):
            edge_density = float(np.mean(canny[candidate] > 0))
            texture_energy = float(np.mean(np.abs(cv2.Laplacian(gray, cv2.CV_32F)[candidate])))
        else:
            edge_density = float(np.mean(canny[mid_band] > 0))
            texture_energy = float(np.mean(np.abs(cv2.Laplacian(gray, cv2.CV_32F)[mid_band])))

        left_side = float(np.mean(candidate[:, : w // 3]))
        right_side = float(np.mean(candidate[:, (2 * w) // 3 :]))
        side_presence = normalize((left_side + right_side) / 2.0, 0.06, 0.28)

        line_features = self._line_features(canny, candidate)
        vh_balance = 1.0 - abs(line_features["vertical_ratio"] - line_features["horizontal_ratio"])
        line_score = 0.5 * vh_balance + 0.5 * normalize(line_features["hv_ratio"], 0.20, 0.75)

        building_score = (
            0.28 * normalize(candidate_ratio, 0.10, 0.55)
            + 0.24 * normalize(edge_density, 0.025, 0.16)
            + 0.18 * normalize(texture_energy, 3.0, 20.0)
            + 0.16 * side_presence
            + 0.14 * clamp(line_score)
        )

        return {
            "building_candidate_ratio": candidate_ratio,
            "building_edge_density": edge_density,
            "building_texture_energy": texture_energy,
            "building_side_presence": side_presence,
            "building_hv_line_ratio": line_features["hv_ratio"],
            "building_vh_balance": vh_balance,
            "building_score": clamp(building_score),
            "line_vanish_like": line_features["vanish_like"],
            "line_diagonal_ratio": line_features["diagonal_ratio"],
            "line_horizontal_ratio": line_features["horizontal_ratio"],
            "line_vertical_ratio": line_features["vertical_ratio"],
        }

    def _score_open_scene(
        self,
        masks: Dict[str, np.ndarray],
        canny: np.ndarray,
        building_score: float,
    ) -> Dict[str, float]:
        sky = masks["sky"]
        vegetation = masks["vegetation"]
        road = masks["road"]

        h, _ = sky.shape
        sky_ratio = float(np.mean(sky))
        top_sky_ratio = float(np.mean(sky[: int(0.50 * h), :]))
        vegetation_ratio = float(np.mean(vegetation))
        road_ratio = float(np.mean(road))
        global_edge_density = float(np.mean(canny > 0))

        open_scene_score = (
            0.34 * normalize(top_sky_ratio, 0.18, 0.75)
            + 0.24 * normalize(vegetation_ratio, 0.12, 0.62)
            + 0.18 * normalize(road_ratio, 0.12, 0.55)
            + 0.14 * inverse_normalize(global_edge_density, 0.05, 0.17)
            + 0.10 * inverse_normalize(building_score, 0.28, 0.66)
        )

        return {
            "sky_ratio": sky_ratio,
            "sky_top_ratio": top_sky_ratio,
            "vegetation_ratio": vegetation_ratio,
            "road_ratio": road_ratio,
            "global_edge_density": global_edge_density,
            "open_scene_score": clamp(open_scene_score),
        }

    def _score_road_corridor(
        self,
        masks: Dict[str, np.ndarray],
        line_vanish_like: float,
    ) -> Dict[str, float]:
        road = masks["road"]
        facade = masks["facade_candidate"]
        h, w = road.shape

        center_bottom = road[int(0.55 * h) :, int(0.25 * w) : int(0.75 * w)]
        center_bottom_road_ratio = float(np.mean(center_bottom)) if center_bottom.size else 0.0

        left_facade = float(np.mean(facade[int(0.20 * h) : int(0.90 * h), : int(0.22 * w)]))
        right_facade = float(np.mean(facade[int(0.20 * h) : int(0.90 * h), int(0.78 * w) :]))
        side_facade_ratio = 0.5 * (left_facade + right_facade)

        distant_center_facade = float(
            np.mean(facade[: int(0.35 * h), int(0.35 * w) : int(0.65 * w)])
        )

        corridor_score = (
            0.45 * normalize(center_bottom_road_ratio, 0.08, 0.62)
            + 0.23 * normalize(line_vanish_like, 0.08, 0.55)
            + 0.20 * inverse_normalize(side_facade_ratio, 0.08, 0.35)
            + 0.12 * inverse_normalize(distant_center_facade, 0.04, 0.20)
        )

        return {
            "road_center_bottom_ratio": center_bottom_road_ratio,
            "road_side_facade_ratio": side_facade_ratio,
            "road_distant_center_facade_ratio": distant_center_facade,
            "road_corridor_score": clamp(corridor_score),
        }

    def _score_occlusion(self, masks: Dict[str, np.ndarray], gray: np.ndarray) -> Dict[str, float]:
        sky = masks["sky"]
        vegetation = masks["vegetation"]
        road = masks["road"]

        h, w = gray.shape
        foreground = (~sky) & (~vegetation) & (~road) & (gray > 30)

        lower = np.zeros_like(foreground, dtype=bool)
        lower[int(0.35 * h) :, :] = True
        obj = (foreground & lower).astype(np.uint8)

        kernel = np.ones((5, 5), np.uint8)
        obj = cv2.morphologyEx(obj, cv2.MORPH_OPEN, kernel, iterations=1)
        obj = cv2.morphologyEx(obj, cv2.MORPH_CLOSE, kernel, iterations=1)

        largest_ratio = 0.0
        num, _, stats, centroids = cv2.connectedComponentsWithStats(obj, connectivity=8)
        if num > 1:
            for i in range(1, num):
                area = int(stats[i, cv2.CC_STAT_AREA])
                if area <= 0:
                    continue
                cx = float(centroids[i, 0])
                cy = float(centroids[i, 1])
                if cy < 0.45 * h:
                    continue
                if not (0.20 * w <= cx <= 0.80 * w):
                    continue
                largest_ratio = max(largest_ratio, area / float(h * w))

        center_region = obj[int(0.45 * h) :, int(0.20 * w) : int(0.80 * w)]
        center_cover_ratio = float(np.mean(center_region > 0)) if center_region.size else 0.0

        occlusion_score = (
            0.55 * normalize(largest_ratio, 0.04, 0.30)
            + 0.45 * normalize(center_cover_ratio, 0.07, 0.50)
        )

        return {
            "occlusion_largest_component_ratio": largest_ratio,
            "occlusion_center_cover_ratio": center_cover_ratio,
            "occlusion_score": clamp(occlusion_score),
        }

    def _score_logo_presence(self, gray: np.ndarray) -> Dict[str, float]:
        if not self.logo_templates:
            return {
                "logo_templates_available": 0.0,
                "logo_match_score": 0.5,
            }

        h, w = gray.shape
        left_crop = gray[int(0.82 * h) :, : int(0.36 * w)]
        right_crop = gray[int(0.82 * h) :, int(0.64 * w) :]

        max_val = 0.0
        for templ in self.logo_templates:
            max_val = max(max_val, safe_template_match(left_crop, templ))
            max_val = max(max_val, safe_template_match(right_crop, templ))

        return {
            "logo_templates_available": 1.0,
            "logo_match_score": clamp(max_val),
        }

    def _score_placeholder_similarity(self, gray: np.ndarray) -> Dict[str, float]:
        if not self.placeholder_refs:
            return {
                "placeholder_similarity": 0.0,
                "placeholder_best_reference": "",
            }

        d_hash = compute_dhash(gray)
        p_hash = compute_phash(gray)

        best_similarity = 0.0
        best_ref = ""

        for ref in self.placeholder_refs:
            sim_d = hash_similarity(d_hash, ref["dhash"])
            sim_p = hash_similarity(p_hash, ref["phash"])
            sim_corr = corr_similarity(gray, ref["gray"])
            sim = 0.45 * sim_p + 0.35 * sim_d + 0.20 * sim_corr
            if sim > best_similarity:
                best_similarity = sim
                best_ref = ref["path"]

        return {
            "placeholder_similarity": clamp(best_similarity),
            "placeholder_best_reference": best_ref,
        }

    def _score_non_google(self, gray: np.ndarray, logo_score: float, logo_available: bool) -> Dict[str, float]:
        h, w = gray.shape

        strip = max(2, int(round(0.05 * min(h, w))))
        strips = [
            gray[:strip, :],
            gray[h - strip :, :],
            gray[:, :strip],
            gray[:, w - strip :],
        ]
        border_extreme_ratio = float(
            np.mean(
                np.concatenate([((x < 15) | (x > 245)).reshape(-1) for x in strips], axis=0)
            )
        )

        aspect_ratio = w / float(max(h, 1))
        aspect_score = normalize(abs(aspect_ratio - 1.0), 0.35, 1.0)

        logo_absence = (1.0 - logo_score) if logo_available else 0.0

        non_google_score = (
            0.45 * normalize(border_extreme_ratio, 0.05, 0.45)
            + 0.25 * aspect_score
            + 0.30 * logo_absence
        )

        return {
            "border_extreme_ratio": border_extreme_ratio,
            "aspect_ratio": aspect_ratio,
            "non_google_score": clamp(non_google_score),
        }

    def _score_interior(
        self,
        hsv: np.ndarray,
        building_score: float,
        sky_ratio: float,
        vegetation_ratio: float,
        center_road_ratio: float,
        logo_score: float,
        logo_available: bool,
        line_horizontal_ratio: float,
    ) -> Dict[str, float]:
        h = hsv[:, :, 0]
        s = hsv[:, :, 1]
        v = hsv[:, :, 2]

        warm_ratio = float(np.mean((h <= 25) & (s >= 35) & (v >= 35)))

        outdoor_evidence = clamp(
            0.40 * building_score
            + 0.25 * sky_ratio
            + 0.20 * center_road_ratio
            + 0.15 * vegetation_ratio
        )

        logo_absence = (1.0 - logo_score) if logo_available else 0.0

        interior_score = (
            0.48 * inverse_normalize(outdoor_evidence, 0.18, 0.58)
            + 0.22 * normalize(warm_ratio, 0.12, 0.55)
            + 0.16 * normalize(line_horizontal_ratio, 0.25, 0.85)
            + 0.14 * logo_absence
        )

        return {
            "interior_warm_ratio": warm_ratio,
            "interior_outdoor_evidence": outdoor_evidence,
            "interior_score": clamp(interior_score),
        }

    def analyze(self, image_bgr: np.ndarray, image_path: Path) -> Tuple[Dict[str, float], Dict[str, np.ndarray]]:
        image_bgr, _ = resize_keep_aspect(image_bgr, self.config.runtime.max_side)
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
        canny = cv2.Canny(gray, 80, 180)

        masks = self._compute_masks(hsv, gray)

        scores: Dict[str, float] = {}

        dark = self._score_darkness(hsv[:, :, 2])
        scores.update(dark)

        glare = self._score_glare(image_bgr, hsv, gray)
        scores.update(glare)

        blur = self._score_blur(gray, canny, masks["facade_candidate"])
        scores.update(blur)

        building = self._score_building_presence(gray, canny, masks["facade_candidate"])
        scores.update(building)

        open_scene = self._score_open_scene(masks, canny, scores["building_score"])
        scores.update(open_scene)

        corridor = self._score_road_corridor(masks, scores["line_vanish_like"])
        scores.update(corridor)

        occlusion = self._score_occlusion(masks, gray)
        scores.update(occlusion)

        logo = self._score_logo_presence(gray)
        scores.update(logo)

        placeholder = self._score_placeholder_similarity(gray)
        scores.update(placeholder)

        non_google = self._score_non_google(
            gray,
            scores["logo_match_score"],
            bool(scores["logo_templates_available"] > 0.5),
        )
        scores.update(non_google)

        interior = self._score_interior(
            hsv=hsv,
            building_score=scores["building_score"],
            sky_ratio=scores["sky_ratio"],
            vegetation_ratio=scores["vegetation_ratio"],
            center_road_ratio=scores["road_center_bottom_ratio"],
            logo_score=scores["logo_match_score"],
            logo_available=bool(scores["logo_templates_available"] > 0.5),
            line_horizontal_ratio=scores["line_horizontal_ratio"],
        )
        scores.update(interior)

        scores["image_height"] = float(image_bgr.shape[0])
        scores["image_width"] = float(image_bgr.shape[1])

        return scores, {"image_bgr": image_bgr, **masks}


In [ ]:
# ----------------------------
# Decision policy + pipeline
# ----------------------------

class DecisionEngine:
    def __init__(self, config: FilterConfig):
        self.config = config

    def decide(self, scores: Dict[str, float]) -> Dict[str, Any]:
        t = self.config.thresholds
        w = self.config.weights

        hard_reasons: List[str] = []
        soft_reasons: List[str] = []

        # Hard rejects
        if scores["placeholder_similarity"] >= t.placeholder_hard_similarity:
            hard_reasons.append("placeholder_no_imagery")

        if (
            scores["glare_score"] >= t.glare_hard_score
            or scores["bright_pixel_ratio"] >= t.bright_hard_pixel_ratio
            or scores["clip_ratio"] >= t.clip_hard_ratio
        ):
            hard_reasons.append("extreme_glare_or_washout")

        if (
            scores["brightness_mean_v"] <= t.dark_hard_mean_v
            and scores["dark_score"] >= t.dark_hard_score
        ):
            hard_reasons.append("extreme_darkness")

        if (
            scores["interior_score"] >= t.interior_hard
            and scores["interior_outdoor_evidence"] <= t.interior_outdoor_evidence_hard_max
        ):
            hard_reasons.append("strong_interior_likelihood")

        if (
            scores["blur_severity"] >= t.blur_hard_severity
            and scores["building_score"] <= t.building_soft_score
        ):
            hard_reasons.append("severe_blur_low_facade_detail")

        low_building = clamp(
            (t.building_min_score - scores["building_score"]) / max(t.building_min_score, 1e-6)
        )

        open_scene_pressure = clamp(
            0.50 * normalize(scores["open_scene_score"], t.open_scene_combined, 1.0)
            + 0.25 * normalize(scores["sky_ratio"], t.sky_dominance, 1.0)
            + 0.25 * normalize(scores["vegetation_ratio"], t.vegetation_dominance, 1.0)
        )

        contributions = {
            "low_building": w.low_building * low_building,
            "open_scene": w.open_scene * open_scene_pressure,
            "blur": w.blur * normalize(scores["blur_severity"], t.blur_soft_severity, 1.0),
            "road_corridor": w.road_corridor * normalize(scores["road_corridor_score"], t.road_corridor, 1.0),
            "occlusion": w.occlusion * normalize(scores["occlusion_score"], t.occlusion_soft, 1.0),
            "interior": w.interior * normalize(scores["interior_score"], t.interior_soft, 1.0),
            "non_google": w.non_google * normalize(scores["non_google_score"], t.non_google_soft, 1.0),
            "glare": w.glare * normalize(scores["glare_score"], t.glare_soft_score, 1.0),
            "dark": w.dark * normalize(scores["dark_score"], t.dark_soft_score, 1.0),
        }

        synergy_bonus = 0.0

        if low_building > 0.35 and open_scene_pressure > 0.45:
            synergy_bonus += 0.10
            soft_reasons.append("low_facade_plus_open_scene")

        if low_building > 0.35 and scores["road_corridor_score"] >= t.road_corridor:
            synergy_bonus += 0.08
            soft_reasons.append("street_corridor_not_facade_view")

        if low_building > 0.30 and scores["vegetation_ratio"] >= t.vegetation_dominance:
            synergy_bonus += 0.06
            soft_reasons.append("vegetation_dominance_low_facade")

        if low_building > 0.25 and scores["blur_severity"] >= t.blur_soft_severity:
            synergy_bonus += 0.07
            soft_reasons.append("blur_with_low_facade_detail")

        if low_building > 0.25 and scores["occlusion_score"] >= t.occlusion_soft:
            synergy_bonus += 0.07
            soft_reasons.append("occlusion_with_low_facade_visibility")

        if scores["placeholder_similarity"] >= t.placeholder_soft_similarity:
            soft_reasons.append("placeholder_similarity_high")

        if scores["interior_score"] >= t.interior_soft and scores["interior_outdoor_evidence"] <= t.interior_outdoor_evidence_soft_max:
            soft_reasons.append("possible_interior")

        invalid_score = clamp(sum(contributions.values()) + synergy_bonus)

        hard_reject = len(hard_reasons) > 0
        is_invalid = hard_reject or invalid_score >= t.invalid_score_threshold

        rescued_by_building = False
        if (
            is_invalid
            and not hard_reject
            and scores["building_score"] >= (t.building_min_score + 0.10)
            and scores["open_scene_score"] < 0.72
            and scores["road_corridor_score"] < 0.72
        ):
            is_invalid = False
            rescued_by_building = True
            soft_reasons.append("rescued_by_building_presence")

        # Build reason list and pick primary
        if hard_reasons:
            reasons = hard_reasons + [r for r in soft_reasons if r not in hard_reasons]
            primary_reason = hard_reasons[0]
        else:
            reasons = soft_reasons.copy()
            if not reasons:
                reasons = ["valid_facade_view"]

            if is_invalid:
                reason_map = {
                    "low_building": "insufficient_facade_presence",
                    "open_scene": "open_scene_dominance",
                    "road_corridor": "street_corridor_view",
                    "blur": "blurred_facade",
                    "occlusion": "vehicle_or_object_occlusion",
                    "interior": "interior_likelihood",
                    "glare": "glare_or_overexposure",
                    "dark": "dark_or_night",
                    "non_google": "non_google_format_signal",
                }
                top_component = max(contributions.items(), key=lambda kv: kv[1])[0]
                primary_reason = reason_map.get(top_component, reasons[0])
            else:
                primary_reason = "valid"

        borderline = abs(invalid_score - t.invalid_score_threshold) <= t.borderline_margin

        if hard_reject:
            confidence = clamp(max(
                scores["placeholder_similarity"],
                scores["glare_score"],
                scores["dark_score"],
                scores["interior_score"],
                scores["blur_severity"],
            ))
        else:
            confidence = clamp(0.5 + abs(invalid_score - t.invalid_score_threshold) * 0.9)

        return {
            "is_invalid": bool(is_invalid),
            "hard_reject": bool(hard_reject),
            "borderline": bool(borderline),
            "invalid_score": float(invalid_score),
            "confidence": float(confidence),
            "primary_reason": primary_reason,
            "reasons": reasons,
            "secondary_reasons": [r for r in reasons if r != primary_reason],
            "rescued_by_building": bool(rescued_by_building),
            "contrib_low_building": float(contributions["low_building"]),
            "contrib_open_scene": float(contributions["open_scene"]),
            "contrib_blur": float(contributions["blur"]),
            "contrib_road_corridor": float(contributions["road_corridor"]),
            "contrib_occlusion": float(contributions["occlusion"]),
            "contrib_interior": float(contributions["interior"]),
            "contrib_non_google": float(contributions["non_google"]),
            "contrib_glare": float(contributions["glare"]),
            "contrib_dark": float(contributions["dark"]),
            "synergy_bonus": float(synergy_bonus),
        }


class StreetViewInvalidFilterPipeline:
    def __init__(self, input_folder: Path, output_folder: Path, config: FilterConfig):
        self.input_folder = Path(input_folder)
        self.output_folder = Path(output_folder)
        self.config = config

        self.detectors = MultiCueDetectors(config)
        self.engine = DecisionEngine(config)

        self.logger = logging.getLogger("sv_filter")
        self.logger.setLevel(getattr(logging, config.runtime.log_level.upper(), logging.INFO))
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
            self.logger.addHandler(handler)

        random.seed(config.runtime.random_seed)
        np.random.seed(config.runtime.random_seed)

        self.reports_dir = self.output_folder / "reports"
        self.invalid_dir = self.output_folder / "invalid"
        self.debug_dir = self.output_folder / "debug"
        self.review_dir = self.output_folder / "review_samples"

        ensure_dir(self.output_folder)
        ensure_dir(self.reports_dir)
        ensure_dir(self.invalid_dir)
        ensure_dir(self.debug_dir)
        ensure_dir(self.review_dir)

        self.results_jsonl_path = self.reports_dir / "results_v3.jsonl"
        self.results_csv_path = self.reports_dir / "results_v3.csv"
        self.results_json_path = self.reports_dir / "results_v3.json"
        self.summary_json_path = self.reports_dir / "summary_v3.json"
        self.config_json_path = self.reports_dir / "effective_config_v3.json"

        self.debug_reason_counts: Dict[str, int] = {}

    def _load_existing_records(self) -> Dict[str, Dict[str, Any]]:
        records: Dict[str, Dict[str, Any]] = {}
        if not self.config.runtime.resume:
            return records
        if not self.results_jsonl_path.exists():
            return records

        with self.results_jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as e:
                    self.logger.warning(
                        "Skipping malformed JSONL record in %s: %s",
                        self.results_jsonl_path,
                        e,
                    )
                    continue
                path_key = row.get("image_path")
                if path_key:
                    records[path_key] = row
        return records

    def _write_jsonl_rows(self, rows: List[Dict[str, Any]]) -> None:
        if not self.config.runtime.write_jsonl or not rows:
            return
        mode = "a" if self.results_jsonl_path.exists() else "w"
        with self.results_jsonl_path.open(mode, encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=True) + "\n")

    def _route_invalid_image(self, src: Path, primary_reason: str) -> Optional[Path]:
        action = self.config.runtime.invalid_action.lower().strip()
        if action == "none":
            return None

        reason_dir = self.invalid_dir / primary_reason if self.config.runtime.organize_by_primary_reason else self.invalid_dir

        if self.config.runtime.preserve_relative_structure:
            try:
                rel_parent = src.relative_to(self.input_folder).parent
            except ValueError:
                rel_parent = Path("")
            reason_dir = reason_dir / rel_parent

        ensure_dir(reason_dir)
        dest = reason_dir / src.name

        if dest.exists():
            stem = dest.stem
            suffix = dest.suffix
            counter = 1
            while True:
                candidate = reason_dir / f"{stem}_{counter}{suffix}"
                if not candidate.exists():
                    dest = candidate
                    break
                counter += 1

        if action == "copy":
            shutil.copy2(src, dest)
        elif action == "move":
            shutil.move(str(src), str(dest))
        else:
            return None

        return dest

    def _maybe_write_debug(self, image_bgr: np.ndarray, row: Dict[str, Any]) -> None:
        if not self.config.runtime.save_debug_images:
            return

        reason = row["primary_reason"] if row["is_invalid"] else "valid"

        if row["is_invalid"]:
            count = self.debug_reason_counts.get(reason, 0)
            if count >= self.config.runtime.debug_max_per_reason:
                return
            self.debug_reason_counts[reason] = count + 1
        else:
            if not row["borderline"] or not self.config.runtime.debug_save_valid_borderline:
                return
            count = self.debug_reason_counts.get("borderline_valid", 0)
            if count >= self.config.runtime.debug_max_per_reason:
                return
            self.debug_reason_counts["borderline_valid"] = count + 1
            reason = "borderline_valid"

        overlay = draw_debug_overlay(
            image_bgr,
            decision={
                "is_invalid": row["is_invalid"],
                "primary_reason": row["primary_reason"],
                "confidence": row["confidence"],
                "invalid_score": row["invalid_score"],
                "hard_reject": row["hard_reject"],
                "borderline": row["borderline"],
                "reasons": row["reasons"].split(",") if row["reasons"] else [],
            },
            scores={
                "building_score": row["building_score"],
                "open_scene_score": row["open_scene_score"],
                "road_corridor_score": row["road_corridor_score"],
                "blur_severity": row["blur_severity"],
                "glare_score": row["glare_score"],
                "dark_score": row["dark_score"],
                "interior_score": row["interior_score"],
                "occlusion_score": row["occlusion_score"],
                "placeholder_similarity": row["placeholder_similarity"],
            },
        )

        out_dir = self.debug_dir / reason
        ensure_dir(out_dir)
        out_path = out_dir / Path(row["image_path"]).name
        cv2.imwrite(str(out_path), overlay)

    def _row_from_scores(
        self,
        image_path: Path,
        decision: Dict[str, Any],
        scores: Dict[str, float],
        routed_path: Optional[Path],
    ) -> Dict[str, Any]:
        t = self.config.thresholds

        row: Dict[str, Any] = {
            "image_path": str(image_path),
            "filename": image_path.name,
            "is_invalid": bool(decision["is_invalid"]),
            "hard_reject": bool(decision["hard_reject"]),
            "borderline": bool(decision["borderline"]),
            "invalid_score": float(decision["invalid_score"]),
            "confidence": float(decision["confidence"]),
            "primary_reason": str(decision["primary_reason"]),
            "secondary_reasons": ",".join(decision["secondary_reasons"]),
            "reasons": ",".join(decision["reasons"]),
            "rescued_by_building": bool(decision["rescued_by_building"]),
            "routed_invalid_path": str(routed_path) if routed_path else "",
            "processed_at": dt.datetime.utcnow().isoformat() + "Z",

            # detector scores
            "brightness_mean_v": float(scores["brightness_mean_v"]),
            "dark_score": float(scores["dark_score"]),
            "bright_pixel_ratio": float(scores["bright_pixel_ratio"]),
            "clip_ratio": float(scores["clip_ratio"]),
            "glare_score": float(scores["glare_score"]),
            "laplacian_var_roi": float(scores["laplacian_var_roi"]),
            "tenengrad_mean_roi": float(scores["tenengrad_mean_roi"]),
            "edge_density_roi": float(scores["edge_density_roi"]),
            "blur_severity": float(scores["blur_severity"]),
            "building_candidate_ratio": float(scores["building_candidate_ratio"]),
            "building_edge_density": float(scores["building_edge_density"]),
            "building_texture_energy": float(scores["building_texture_energy"]),
            "building_side_presence": float(scores["building_side_presence"]),
            "building_hv_line_ratio": float(scores["building_hv_line_ratio"]),
            "building_score": float(scores["building_score"]),
            "sky_ratio": float(scores["sky_ratio"]),
            "sky_top_ratio": float(scores["sky_top_ratio"]),
            "vegetation_ratio": float(scores["vegetation_ratio"]),
            "road_ratio": float(scores["road_ratio"]),
            "open_scene_score": float(scores["open_scene_score"]),
            "road_center_bottom_ratio": float(scores["road_center_bottom_ratio"]),
            "road_side_facade_ratio": float(scores["road_side_facade_ratio"]),
            "road_corridor_score": float(scores["road_corridor_score"]),
            "interior_warm_ratio": float(scores["interior_warm_ratio"]),
            "interior_outdoor_evidence": float(scores["interior_outdoor_evidence"]),
            "interior_score": float(scores["interior_score"]),
            "occlusion_largest_component_ratio": float(scores["occlusion_largest_component_ratio"]),
            "occlusion_center_cover_ratio": float(scores["occlusion_center_cover_ratio"]),
            "occlusion_score": float(scores["occlusion_score"]),
            "non_google_score": float(scores["non_google_score"]),
            "logo_match_score": float(scores["logo_match_score"]),
            "placeholder_similarity": float(scores["placeholder_similarity"]),
            "placeholder_best_reference": str(scores.get("placeholder_best_reference", "")),

            # contributions
            "contrib_low_building": float(decision["contrib_low_building"]),
            "contrib_open_scene": float(decision["contrib_open_scene"]),
            "contrib_blur": float(decision["contrib_blur"]),
            "contrib_road_corridor": float(decision["contrib_road_corridor"]),
            "contrib_occlusion": float(decision["contrib_occlusion"]),
            "contrib_interior": float(decision["contrib_interior"]),
            "contrib_non_google": float(decision["contrib_non_google"]),
            "contrib_glare": float(decision["contrib_glare"]),
            "contrib_dark": float(decision["contrib_dark"]),
            "synergy_bonus": float(decision["synergy_bonus"]),

            # key thresholds used
            "threshold_invalid_score": float(t.invalid_score_threshold),
            "threshold_building_min": float(t.building_min_score),
            "threshold_glare_hard": float(t.glare_hard_score),
            "threshold_blur_hard": float(t.blur_hard_severity),
            "threshold_placeholder_hard": float(t.placeholder_hard_similarity),
            "threshold_interior_hard": float(t.interior_hard),
        }
        return row

    def generate_review_samples(self, df: pd.DataFrame) -> pd.DataFrame:
        cfg = self.config.runtime
        rng = np.random.default_rng(cfg.random_seed)

        picks: List[pd.DataFrame] = []

        invalid_df = df[df["is_invalid"] == True]
        if not invalid_df.empty:
            n = min(cfg.review_random_invalid_count, len(invalid_df))
            idx = rng.choice(invalid_df.index.to_numpy(), size=n, replace=False)
            picks.append(invalid_df.loc[idx].assign(review_bucket="random_invalid"))

        borderline_df = df[df["borderline"] == True]
        if not borderline_df.empty:
            n = min(cfg.review_borderline_count, len(borderline_df))
            idx = rng.choice(borderline_df.index.to_numpy(), size=n, replace=False)
            picks.append(borderline_df.loc[idx].assign(review_bucket="borderline"))

        if not invalid_df.empty:
            for reason, group in invalid_df.groupby("primary_reason"):
                n = min(cfg.review_per_reason_count, len(group))
                idx = rng.choice(group.index.to_numpy(), size=n, replace=False)
                picks.append(group.loc[idx].assign(review_bucket=f"reason_{reason}"))

        if picks:
            review_df = pd.concat(picks, ignore_index=True).drop_duplicates(subset=["image_path", "review_bucket"])
        else:
            review_df = pd.DataFrame(columns=list(df.columns) + ["review_bucket"])

        ensure_dir(self.review_dir)
        review_manifest = self.review_dir / "review_manifest_v3.csv"
        review_df.to_csv(review_manifest, index=False)

        if cfg.review_copy_images and not review_df.empty:
            for _, row in review_df.iterrows():
                src = Path(row["image_path"])
                if not src.exists():
                    continue
                bucket_dir = self.review_dir / row["review_bucket"]
                ensure_dir(bucket_dir)
                dst = bucket_dir / src.name
                if not dst.exists():
                    shutil.copy2(src, dst)

        self.logger.info("Review manifest saved: %s (%d rows)", review_manifest, len(review_df))
        return review_df

    def run(self, max_images: Optional[int] = None) -> pd.DataFrame:
        if not self.input_folder.exists():
            raise FileNotFoundError(f"Input folder does not exist: {self.input_folder}")

        ensure_dir(self.reports_dir)
        self.config_json_path.write_text(json.dumps(self.config.to_dict(), indent=2))

        existing_map = self._load_existing_records()
        processed_paths = set(existing_map.keys()) if self.config.runtime.resume else set()

        image_paths = discover_images(self.input_folder, self.config.runtime)
        if max_images is not None:
            image_paths = image_paths[:max_images]

        self.logger.info("Discovered %d images", len(image_paths))
        self.logger.info("Resume enabled: %s (already processed: %d)", self.config.runtime.resume, len(processed_paths))

        rows_to_append: List[Dict[str, Any]] = []
        processed_count = 0
        skipped_count = 0
        failed_read_count = 0
        failed_process_count = 0

        pbar = tqdm(image_paths, desc="Filtering images")
        for idx, image_path in enumerate(pbar, start=1):
            path_key = str(image_path)
            if path_key in processed_paths:
                skipped_count += 1
                continue

            image_bgr = cv2.imread(path_key, cv2.IMREAD_COLOR)
            if image_bgr is None:
                self.logger.warning("Failed to read image (skipping): %s", path_key)
                failed_read_count += 1
                continue

            try:
                scores, aux = self.detectors.analyze(image_bgr, image_path)
                decision = self.engine.decide(scores)

                routed_path = None
                if decision["is_invalid"]:
                    routed_path = self._route_invalid_image(image_path, decision["primary_reason"])

                row = self._row_from_scores(image_path, decision, scores, routed_path)
                rows_to_append.append(row)
                existing_map[path_key] = row

                self._maybe_write_debug(aux["image_bgr"], row)
            except Exception:
                self.logger.exception("Failed to process image: %s", path_key)
                failed_process_count += 1
                continue

            processed_count += 1
            if (
                self.config.runtime.progress_every_n > 0
                and processed_count % self.config.runtime.progress_every_n == 0
            ):
                self.logger.info(
                    "Processed %d new images | skipped %d | last=%s",
                    processed_count,
                    skipped_count,
                    image_path.name,
                )

        self._write_jsonl_rows(rows_to_append)

        all_rows = list(existing_map.values())
        df = pd.DataFrame(all_rows)
        if not df.empty:
            df = df.sort_values("image_path").reset_index(drop=True)

        if self.config.runtime.write_csv:
            df.to_csv(self.results_csv_path, index=False)
        if self.config.runtime.write_json:
            self.results_json_path.write_text(json.dumps(all_rows, indent=2))

        summary = {
            "generated_at_utc": dt.datetime.utcnow().isoformat() + "Z",
            "input_folder": str(self.input_folder),
            "output_folder": str(self.output_folder),
            "images_discovered": int(len(image_paths)),
            "new_processed": int(processed_count),
            "skipped": int(skipped_count),
            "failed_reads": int(failed_read_count),
            "failed_processing": int(failed_process_count),
            "total_records": int(len(df)),
            "invalid_count": int(df["is_invalid"].sum()) if "is_invalid" in df else 0,
            "valid_count": int((~df["is_invalid"]).sum()) if "is_invalid" in df else 0,
            "invalid_rate": float(df["is_invalid"].mean()) if "is_invalid" in df and len(df) else 0.0,
            "hard_reject_count": int(df["hard_reject"].sum()) if "hard_reject" in df else 0,
            "borderline_count": int(df["borderline"].sum()) if "borderline" in df else 0,
            "primary_reason_counts": (
                df["primary_reason"].value_counts().to_dict() if "primary_reason" in df else {}
            ),
            "average_scores": {
                k: float(df[k].mean())
                for k in [
                    "invalid_score",
                    "building_score",
                    "open_scene_score",
                    "blur_severity",
                    "glare_score",
                    "dark_score",
                    "interior_score",
                    "occlusion_score",
                    "placeholder_similarity",
                ]
                if k in df and len(df)
            },
        }

        self.summary_json_path.write_text(json.dumps(summary, indent=2))

        self.logger.info("Run complete. Results CSV: %s", self.results_csv_path)
        self.logger.info("Summary JSON: %s", self.summary_json_path)
        if failed_read_count or failed_process_count:
            self.logger.warning(
                "Completed with failures: %d unreadable image(s), %d processing error(s). "
                "See warnings above for affected files.",
                failed_read_count,
                failed_process_count,
            )

        self.generate_review_samples(df)
        return df


def evaluate_with_labels(
    results_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    results_path_col: str = "image_path",
    labels_path_col: str = "image_path",
    labels_target_col: str = "is_invalid",
) -> pd.DataFrame:
    merged = results_df.merge(
        labels_df[[labels_path_col, labels_target_col]],
        left_on=results_path_col,
        right_on=labels_path_col,
        how="inner",
    )
    if merged.empty:
        print("No overlap between results and labels.")
        return merged

    y_true = merged[labels_target_col].astype(bool)
    y_pred = merged["is_invalid"].astype(bool)

    tp = int(np.sum((y_true == True) & (y_pred == True)))
    tn = int(np.sum((y_true == False) & (y_pred == False)))
    fp = int(np.sum((y_true == False) & (y_pred == True)))
    fn = int(np.sum((y_true == True) & (y_pred == False)))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)

    print(f"Matched labels: {len(merged)}")
    print(f"TP={tp} TN={tn} FP={fp} FN={fn}")
    print(f"Precision={precision:.4f} Recall={recall:.4f} F1={f1:.4f}")

    return merged


In [ ]:
# ----------------------------
# STEP 3 - Entry point: run the pipeline
# ----------------------------

pipeline = StreetViewInvalidFilterPipeline(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    config=CONFIG,
)

# Set max_images=None for full run (recommended for production pass)
# For quick smoke test, set max_images=200
results_df = pipeline.run(max_images=None)

print("\nDone.")
print("Total images in manifest:", len(results_df))
if len(results_df):
    print("Invalid count:", int(results_df["is_invalid"].sum()))
    print("Top primary reasons:")
    print(results_df["primary_reason"].value_counts().head(10))

print("\nOutputs:")
print("-", pipeline.results_csv_path)
print("-", pipeline.results_jsonl_path)
print("-", pipeline.results_json_path)
print("-", pipeline.summary_json_path)
print("-", pipeline.review_dir / "review_manifest_v3.csv")


In [ ]:
# ----------------------------
# STEP 3b - Lightweight detector self-tests
# ----------------------------

def run_detector_self_tests() -> None:
    cfg = FilterConfig()
    det = MultiCueDetectors(cfg)
    eng = DecisionEngine(cfg)

    # 1) Dark frame should score high on dark
    dark_img = np.full((256, 256, 3), 10, dtype=np.uint8)
    s_dark, _ = det.analyze(dark_img, Path("dark.jpg"))
    assert s_dark["dark_score"] > 0.80, f"dark_score too low: {s_dark['dark_score']}"

    # 2) Fully white frame should score high on glare
    bright_img = np.full((256, 256, 3), 255, dtype=np.uint8)
    s_bright, _ = det.analyze(bright_img, Path("bright.jpg"))
    assert s_bright["glare_score"] > 0.80, f"glare_score too low: {s_bright['glare_score']}"

    # 3) Blurred checkerboard should be blurrier than sharp checkerboard
    sharp = np.zeros((256, 256, 3), dtype=np.uint8)
    for x in range(0, 256, 16):
        cv2.line(sharp, (x, 0), (x, 255), (255, 255, 255), 1)
    for y in range(0, 256, 16):
        cv2.line(sharp, (0, y), (255, y), (255, 255, 255), 1)
    blurred = cv2.GaussianBlur(sharp, (17, 17), 0)

    s_sharp, _ = det.analyze(sharp, Path("sharp.jpg"))
    s_blur, _ = det.analyze(blurred, Path("blurred.jpg"))
    assert s_blur["blur_severity"] > s_sharp["blur_severity"], (
        s_blur["blur_severity"],
        s_sharp["blur_severity"],
    )

    # 4) Placeholder matching sanity (self-similarity)
    ref_gray = cv2.cvtColor(bright_img, cv2.COLOR_BGR2GRAY)
    det.placeholder_refs = [
        {
            "path": "synthetic_ref",
            "gray": ref_gray,
            "dhash": compute_dhash(ref_gray),
            "phash": compute_phash(ref_gray),
        }
    ]
    s_ph, _ = det.analyze(bright_img, Path("same_as_placeholder.jpg"))
    assert s_ph["placeholder_similarity"] > 0.95, s_ph["placeholder_similarity"]

    # 5) Open scene + low facade should trend toward invalid/no-facade
    scene = np.full((256, 256, 3), (230, 230, 200), dtype=np.uint8)
    scene[130:, :] = (60, 150, 60)
    s_scene, _ = det.analyze(scene, Path("open_scene.jpg"))
    d_scene = eng.decide(s_scene)
    assert s_scene["open_scene_score"] > 0.60
    assert d_scene["is_invalid"] is True

    print("All detector self-tests passed.")


# Uncomment to run sanity tests:
# run_detector_self_tests()



## STEP 6 - Threshold tuning quick guide (first 20k rerun)

1. Start with `invalid_action='copy'` and inspect `reports/review_manifest_v3.csv` plus debug overlays.
2. If too many valid facades are rejected for low-building/open-scene:
   - decrease `thresholds.building_min_score` by ~0.02
   - increase `thresholds.invalid_score_threshold` by ~0.02
3. If obvious corridor/no-facade images are slipping through:
   - decrease `thresholds.road_corridor`
   - increase `weights.road_corridor` slightly (e.g., +0.02)
4. If blur misses obvious blurry buildings:
   - decrease `thresholds.blur_soft_severity`
   - keep `blur_hard_severity` conservative to avoid overfiring
5. For placeholder/no-imagery misses:
   - add more reference placeholders to `placeholder_reference_paths`
   - lower `placeholder_soft_similarity` modestly (e.g., 0.88 -> 0.85)
6. Keep `non_google` as a weak/demoted signal unless you have strong templates and evidence.
